# Recuperar y ordenar productos

Combina BM25, `multilingual-e5-base` y `bge-reranker-v2-m3`. Ajusta α y β con consultas `dev` y compara variantes con las mismas consultas `eval`. Las etiquetas ESCI solo alimentan las métricas.

In [2]:
#%pip -q install -r requirements-lab4-modular.txt

In [3]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src" / "lab4_asistente").is_dir():
    ROOT = next((p for p in ROOT.parents if (p / "src" / "lab4_asistente").is_dir()), ROOT)
sys.path.insert(0, str(ROOT / "src"))
print("Raíz del proyecto:", ROOT)
import json
import numpy as np
import pandas as pd
import torch
from lab4_asistente import config as C
from lab4_asistente.recuperacion import Recuperador, precomputar, ajustar, evaluar, evaluar_catalogo


Raíz del proyecto: /home/winzcode/proyectos/ADEEP_LEARNING/asistente-compras-por-voz


## Cargar catálogo, consultas y etiquetas

In [4]:
catalogo = pd.read_parquet(C.RUTA_DATOS / "catalogo.parquet")
consultas = pd.read_parquet(C.RUTA_DATOS / "consultas.parquet")
qrels = pd.read_parquet(C.RUTA_EVAL / "qrels.parquet")
dev = consultas[consultas.conjunto == "dev"].reset_index(drop=True)
eval_final = consultas[consultas.conjunto == "eval"].reset_index(drop=True)
qrels_dev = qrels[qrels.query_id.isin(dev.query_id)]
qrels_eval = qrels[qrels.query_id.isin(eval_final.query_id)]
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"{len(catalogo):,} productos | {len(dev)} consultas dev | {len(eval_final)} consultas eval | {DEVICE}")


10,688 productos | 150 consultas dev | 250 consultas eval | cuda


## Construir recuperación BM25 + e5 + reranker

In [5]:
rec = Recuperador(catalogo)
rec.cargar_denso(C.MODELO_EMB, DEVICE, C.RUTA_CACHE)
rec.cargar_reranker(C.MODELO_RERANK, DEVICE)
print("Recuperador listo")


/home/winzcode/proyectos/ADEEP_LEARNING/asistente-compras-por-voz/.venv-dl-cuda/lib/python3.11/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


Recuperador listo


## Ajustar α y β en desarrollo

In [6]:
pre_dev = precomputar(rec, dev, qrels_dev, C.GANANCIAS)
parametros, tabla_ajuste = ajustar(pre_dev, C.K_NDCG, C.GRID)
print("Parámetros elegidos con dev:", parametros)
display(tabla_ajuste.sort_values("ndcg_dev", ascending=False).head(10))
rec.alpha = parametros.get("alpha", 0.7)
rec.beta = parametros.get("beta_sistema", 0.8)
C.RUTA_RESULTADOS.mkdir(parents=True, exist_ok=True)
parametros_path = C.RUTA_RESULTADOS / "recuperacion_parametros.json"
parametros_path.write_text(json.dumps(parametros, indent=2), encoding="utf-8")


Parámetros elegidos con dev: {'alpha': 0.7, 'beta_sistema': 0.8, 'beta_bm25+reranker': 1.0}


,metodo,alpha,beta,ndcg_dev
19,sistema,0.7,0.8,0.759112
18,sistema,0.7,0.7,0.758531
20,sistema,0.7,0.9,0.757424
32,bm25+reranker,0.7,1.0,0.756798
21,sistema,0.7,1.0,0.756798
17,sistema,0.7,0.6,0.754017
31,bm25+reranker,0.7,0.9,0.753605
16,sistema,0.7,0.5,0.750977
30,bm25+reranker,0.7,0.8,0.749487
14,sistema,0.7,0.3,0.748407


70

## Comparar recuperadores

Mide BM25, el sistema completo y las variantes sin reranker o sin búsqueda densa sobre las mismas consultas. Aquí no se ajustan parámetros.

In [7]:
pre_eval = precomputar(rec, eval_final, qrels_eval, C.GANANCIAS)
variantes = ["bm25", "sistema", "hibrido", "bm25+reranker"]
filas = []
for metodo in variantes:
    beta = parametros.get(f"beta_{metodo}", rec.beta)
    tabla = evaluar(pre_eval, metodo, rec.alpha, beta, C.K_NDCG)
    filas.append({"variante": metodo, "nDCG@10 candidatos anotados": tabla.ndcg.mean()})

# Métrica complementaria usando la recuperación sobre el catálogo completo.
for metodo in variantes:
    beta = parametros.get(f"beta_{metodo}", rec.beta)
    full = evaluar_catalogo(rec, eval_final, qrels_eval, metodo, C.GANANCIAS, C.K_NDCG, beta)
    filas[variantes.index(metodo)]["nDCG@10 catálogo completo"] = full.ndcg.mean()

resultados_rec = pd.DataFrame(filas)
display(resultados_rec)
C.RUTA_RESULTADOS.mkdir(parents=True, exist_ok=True)
resultados_rec.to_csv(C.RUTA_RESULTADOS / "recuperacion_eval.csv", index=False)


,variante,nDCG@10 candidatos anotados,nDCG@10 catálogo completo
0,bm25,0.692541,0.562320
1,sistema,0.765444,0.685105
2,hibrido,0.732333,0.633484
3,bm25+reranker,0.769171,0.656652


## Ejemplo de productos recuperados

In [8]:
query = eval_final.iloc[0].query
ids = rec.buscar(query, k=C.K_RESULTADOS, metodo="sistema")
display(catalogo.iloc[ids][["product_id", "titulo", "marca", "color"]])
print("Consulta:", query)


,product_id,titulo,marca,color
1776,B010ON3NHO,Salerm Cosmetics Champú Purificante - 1000 ml,Salerm,Único
785,B00684D55Y,Salerm Cosmetics Champú Equilibrium - 250 ml,Salerm,
1554,B00TBTWDX8,Salerm Cosmetics Hi Repair Champú - 250 ml,Salerm,Blanco
1448,B00O79V522,SALERM CHAMPU Equilibrante 5100ml Equilibrium,Salerm,
812,B0075CMHSE,"Salerm Cosmetics Dermocalm Champú - 250 ml, Tr...",Salerm,Transparente


Consulta: , champú sin sal salerm


## Batería de pruebas: consultas y resultados

Ejecuta cinco consultas representativas. Para cada una, compara el tipo de producto esperado con los productos recuperados por el sistema. El estado indica si la expectativa aparece entre los primeros resultados.

In [9]:
# Edita las consultas o expectativas para probar otros casos.
casos_prueba = [
    {"consulta": "audífonos inalámbricos bluetooth", "esperado": "audífonos inalámbricos"},
    {"consulta": "mochila resistente para laptop de 15 pulgadas", "esperado": "mochila para laptop"},
    {"consulta": "cafetera de goteo para preparar café", "esperado": "cafetera"},
    {"consulta": "zapatillas deportivas para correr", "esperado": "zapatillas para correr"},
    {"consulta": "botella térmica reutilizable para agua", "esperado": "botella térmica"},
]

def normalizar_prueba(texto):
    return " ".join(str(texto).lower().split())

filas_prueba = []
for caso in casos_prueba:
    indices = rec.buscar(caso["consulta"], k=5, metodo="sistema")
    productos = catalogo.iloc[indices][["product_id", "titulo", "marca", "color"]].copy()
    titulos = productos["titulo"].fillna("").tolist()
    esperado = normalizar_prueba(caso["esperado"])
    coincide = any(esperado in normalizar_prueba(titulo) for titulo in titulos)
    filas_prueba.append({
        "Consulta": caso["consulta"],
        "Esperado": caso["esperado"],
        "Resultado real (top 5)": " | ".join(titulos),
        "Estado": "Coincide" if coincide else "Revisar",
    })
    print(f"\nConsulta: {caso['consulta']}")
    print(f"Esperado: {caso['esperado']}")
    display(productos)
tabla_pruebas = pd.DataFrame(filas_prueba)
display(tabla_pruebas)


Consulta: audífonos inalámbricos bluetooth
Esperado: audífonos inalámbricos


,product_id,titulo,marca,color
5598,B07Q79C2D8,"Auriculares Inalámbricos Bluetooth, HOMSCAM Au...",HOMSCAM,Negro
9865,B093GLP7QY,"Auriculares Inalambricos Deportivos, Auricular...",Donerton,Negro
10582,B09FF5C5YK,"Auriculares Bluetooth, Auriculares Inalámbrico...",Ssmnr,Negro
5388,B07P1T5P97,Auriculares Inalámbricos Bluetooth 5.0 SoundPE...,SoundPEATS,Negro
9960,B094MJLRBL,Auriculares inalámbricos Bluetooth 5.0 con can...,AUKOY,Azul



Consulta: mochila resistente para laptop de 15 pulgadas
Esperado: mochila para laptop


,product_id,titulo,marca,color
9877,B093NYPDX5,FUXINGYAO Mochila Unisex para Ordenador Portát...,FUXINGYAO,Negro
3128,B073F8YN67,HSEOK Bolso Bandolera/Maletín para Portátil 15...,HSEOK,Negro
5676,B07QTHKC4G,HOMIEE 15.6 Pulgadas Mochila de Viaje para Por...,HOMIEE,Gris
3244,B074KV9TT4,"Mochila de viaje de negocios, mochila para com...",MATEIN,Negro
5590,B07Q5S8K28,Mochila Unisex Impermeable para Ordenador Port...,Besttravel,Negro



Consulta: cafetera de goteo para preparar café
Esperado: cafetera


,product_id,titulo,marca,color
4428,B07G4P24SP,Nescafé Dolce Gusto De'Longhi EDG240.R Cafeter...,Nescafé Dolce Gusto,Rojo/Negro
4506,B07GQKR88T,Bosch Tassimo Happy TAS1003 - Cafetera de cáps...,Bosch Hogar,Negro/Rojo
1377,B00M9Y1Y6E,Krups Dolce Gusto Oblo KP1108 - Cafetera de cá...,Krups,Negro
5077,B07M8J4ZXK,Philips Cafetera de cápsulas LM8012/60 Espress...,Philips,Negro
4482,B07GJY9KH9,Krups Nescafé Dolce Gusto Mini Me S KP123B Caf...,Krups,Negro Y Blanco



Consulta: zapatillas deportivas para correr
Esperado: zapatillas para correr


,product_id,titulo,marca,color
6625,B07XRLVQ3T,Knixmax-Zapatillas Deportivas de Hombre Zapati...,Knixmax,Tejer Blanco Para Hombres
8018,B08CYZP2SF,Niños Zapatos Deportivos Running Zapatillas Un...,PAMRAY,Caqui
3526,B077MQJS3R,"PUMA Viz Runner, Zapatillas de Running Hombre,...",PUMA,Negro Puma Black Puma White
6914,B07ZTW38C9,"Nike Air MAX 270 React (GS), Zapatillas para C...",NIKE,Negro Black Cerulean Thunder Grey Barely Volt
7569,B0876WP7WH,Zapatillas de Deporte Hombre Mujer Respirable ...,AICARE,8 5 Rojo



Consulta: botella térmica reutilizable para agua
Esperado: botella térmica


,product_id,titulo,marca,color
9462,B08Y339SGZ,KIDYMOM Botella Agua Acero Inoxidable Para Niñ...,KIDYMOM,Dinosaurios
6926,B07ZYTQHY9,"LSTK Juego de Vasos, Sin BPA, Reutilizables y ...",LSTK,
9360,B08X37ZDJV,"Amazon Brand - Eono Botella de Agua Niños, 420...",Eono,Verde
7619,B087TLGKDP,"ML Botella de Agua Infantil, cantimplora Botel...",ML,Colores Muy Vivos Y Alegres.
6922,B07ZVRWJSZ,"LHKJ 8 pcs Vasos de Plástico, Vasos de Whisky,...",LHKJ,


,Consulta,Esperado,Resultado real (top 5),Estado
0,audífonos inalámbricos bluetooth,audífonos inalámbricos,"Auriculares Inalámbricos Bluetooth, HOMSCAM Au...",Revisar
1,mochila resistente para laptop de 15 pulgadas,mochila para laptop,FUXINGYAO Mochila Unisex para Ordenador Portát...,Revisar
2,cafetera de goteo para preparar café,cafetera,Nescafé Dolce Gusto De'Longhi EDG240.R Cafeter...,Coincide
3,zapatillas deportivas para correr,zapatillas para correr,Knixmax-Zapatillas Deportivas de Hombre Zapati...,Coincide
4,botella térmica reutilizable para agua,botella térmica,KIDYMOM Botella Agua Acero Inoxidable Para Niñ...,Revisar
